# Create Model Ready Dataset (Facsimile Example)

This notebook prepares a model ready dataset from the facsimile data.  
The goal is to transform raw longitudinal records into a structured format that can be directly used for RNN environment training and Q-learning.

Steps include:

1. Constructing time-dependent features (e.g., months since last vaccination)
2. Discretizing variables for RL state representation
3. Creating dummy variables for categorical inputs
4. Combining all components into a final dataframe

In [1]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import pickle
%matplotlib inline


In [ ]:
INPUT_PATH = "facsimile_data.csv"
OUTPUT_MODEL_READY_PATH = "facsimile_model_ready_data.csv"

# Load facsimile long-format dataset
# Each row corresponds to one patient at one time step
df = pd.read_csv(INPUT_PATH)

# Construct time-dependent feature: months since last vaccination
# We compute the number of time steps since the last vaccination event for each patient trajectory
# This is done within each patient (grouped by `id`)
# If no prior vaccination exists, we assign a negative value (e.g., -1)
# This feature will later be discretized and used in RL state construction
def add_months_since_vax(df: pd.DataFrame) -> pd.DataFrame:
    # Loop over each patient trajectory
    pieces = []

    for _, group in df.groupby("id", sort=False):
        group = group.copy().reset_index(drop=True)
        months_since = []
        last_vax_pos = None

        for pos, row in group.iterrows():
            if last_vax_pos is None:
                cur = -1
            else:
                cur = pos - last_vax_pos

            months_since.append(cur)

            if int(row["action"]) == 1:
                last_vax_pos = pos

        group["months_since_vax"] = months_since
        pieces.append(group)

    return pd.concat(pieces, axis=0).reset_index(drop=True)

# Standardize age for RNN input
scaler = StandardScaler()
age_values = np.sort(df["Age.FirstDose"].astype(int).unique()).reshape(-1, 1)
normalized_age = scaler.fit_transform(age_values).reshape(-1)
age_map = dict(zip(age_values.reshape(-1), normalized_age))
df["Age_standardized"] = df["Age.FirstDose"].astype(int).map(age_map)

# Discretization for RL state variables
# Continuous or ordinal variables are discretized into categories to define RL states:
# `age_cat`: age groups
# `months_since_vax_cat`: time since last vaccination
# These categorical variables define the RL state space used in tabular Q-learning.
df["age_cat"] = pd.cut(
    df["Age.FirstDose"].astype(int),
    bins=[0, 18, 30, 50, 65, 100],
    include_lowest=True,
    right=False,
    labels=False,
).astype(int)

# Apply feature construction
df = add_months_since_vax(df)

# Discretize months since vaccination into 3 categories:
# 0–4, 5–6, 7+
df["months_since_vax_cat"] = np.select(
    [
        df["months_since_vax"] < 5,
        (df["months_since_vax"] >= 5) & (df["months_since_vax"] < 7),
        df["months_since_vax"] >= 7,
    ],
    [0, 1, 2],
    default=2,
).astype(int)

# Create dummy variables
# Convert categorical variables into one-hot encoding for the RNN model.
race_dummies = pd.get_dummies(df["Race"], prefix="race")
gender_male = (df["Gender"] == "M").astype(int).rename("gender_male")
variant_dummies = pd.get_dummies(df["variant"], prefix="variant")
visits_dummies = pd.get_dummies(
    pd.cut(
        df["Visits"],
        bins=[0, 5, 10, 20, 50, 1000],
        include_lowest=True,
        right=False
    ),
    prefix="visits"
)
windex_dummies = pd.get_dummies(
    pd.cut(
        df["windex"],
        bins=[0, 1, 3, 5, 100],
        include_lowest=True,
        right=False
    ),
    prefix="windex"
)
age_cat_dummies = pd.get_dummies(df["age_cat"], prefix="agecat")

# Construct final model ready dataset
# We combine original variables, engineered features, and dummy variables into 
# a single dataframe that can be directly used by the learning pipeline.
model_ready = pd.concat(
    [
        df[
            [
                "id",
                "action",
                "Age.FirstDose",
                "Visits",
                "imm_baseline",
                "windex",
                "numVax",
                "severe_infection_next",
                "inf_next",
                "months_since_vax",
                "months_since_vax_cat",
                "age_cat",
            ]
        ],
        race_dummies,
        variant_dummies,
        gender_male,
        age_cat_dummies,
    ],
    axis=1,
)


In [ ]:
print(model_ready.columns.tolist())
print(model_ready.shape)

# output the model ready dataset for use in the learning pipeline
model_ready.to_csv(OUTPUT_MODEL_READY_PATH, index=False)


['id', 'action', 'Age.FirstDose', 'Visits', 'imm_baseline', 'windex', 'numVax', 'severe_infection_next', 'inf_next', 'months_since_vax', 'months_since_vax_cat', 'age_cat', 'race_African American', 'race_Caucasian', 'race_Other', 'variant_delta', 'variant_none', 'variant_omicron', 'gender_male', 'agecat_0', 'agecat_1', 'agecat_2', 'agecat_3', 'agecat_4']
(26378, 24)
